<a href="https://www.kaggle.com/code/aabdollahii/7-ml-solution-arsi-qwen-analysis?scriptVersionId=338912904" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

file_path = "/kaggle/input/datasets/aabdollahii/humanvsai/HVA-Qewn-only.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())

print("\nInfo:")
df.info()

print("\nMissing values:")
print(df.isna().sum())

print("\nLabel distribution:")
print(df["label"].value_counts(dropna=False))


In [ ]:
# Show full text without truncation
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 2000)

text_col = "text"
label_col = "label"

print("Shape:", df.shape)
print(df[label_col].value_counts())

# Number of samples per class
n_samples = 10

human_samples = (
    df[df[label_col] == "human"]
    .sample(n=min(n_samples, (df[label_col] == "human").sum()), random_state=42)
    [["year", "filename", "word_count", "label", "source_column", text_col]]
)

machine_samples = (
    df[df[label_col] == "machine"]
    .sample(n=min(n_samples, (df[label_col] == "machine").sum()), random_state=42)
    [["year", "filename", "word_count", "label", "source_column", text_col]]
)

print("\n================ HUMAN SAMPLES ================\n")
display(human_samples)

print("\n================ MACHINE SAMPLES ================\n")
display(machine_samples)

In [ ]:
def print_full_samples(data, title, n=10, random_state=42):
    samples = data.sample(n=min(n, len(data)), random_state=random_state)

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for i, row in samples.iterrows():
        print("\n" + "-" * 100)
        print(f"Index: {i}")
        print(f"Year: {row['year']}")
        print(f"Filename: {row['filename']}")
        print(f"Label: {row['label']}")
        print(f"Source column: {row['source_column']}")
        print(f"Word count: {row['word_count']}")
        print("-" * 100)
        print(row["text"])
        print("-" * 100)

print_full_samples(
    df[df["label"] == "human"],
    title="FULL HUMAN TEXT SAMPLES",
    n=10,
    random_state=42
)

print_full_samples(
    df[df["label"] == "machine"],
    title="FULL MACHINE TEXT SAMPLES",
    n=10,
    random_state=42
)


In [ ]:
import re
import pandas as pd
from hazm import Normalizer

file_path = "/kaggle/input/datasets/aabdollahii/humanvsai/HVA-Qewn-only.csv"

df = pd.read_csv(file_path)

print("Original shape:", df.shape)
print(df["label"].value_counts())

normalizer = Normalizer(
    correct_spacing=True,
    remove_diacritics=True,
    remove_specials_chars=True,
    decrease_repeated_chars=False,
    persian_style=True,
    persian_numbers=True,
    unicodes_replacement=True,
    seperate_mi=True
)

def normalize_text_hazm_full(text):
    text = str(text)

    # Remove invisible direction/control chars that may remain in scraped text
    text = re.sub(r"[\u200b\u200d\u200e\u200f\ufeff]", " ", text)

    # Convert line breaks and tabs to space
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Normalize Arabic/Persian variants and spacing with Hazm
    text = normalizer.normalize(text)

    # Final whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return text

df_clean = df.copy()

df_clean["text_before"] = df_clean["text"]
df_clean["text"] = df_clean["text"].apply(normalize_text_hazm_full)

df_clean["char_count_before"] = df_clean["text_before"].astype(str).str.len()
df_clean["char_count_after"] = df_clean["text"].astype(str).str.len()
df_clean["word_count_before"] = df_clean["text_before"].astype(str).apply(lambda x: len(x.split()))
df_clean["word_count_after"] = df_clean["text"].astype(str).apply(lambda x: len(x.split()))

df_clean["word_count"] = df_clean["word_count_after"]

print("\nStats by label after Hazm normalization:")
display(
    df_clean.groupby("label")[["char_count_after", "word_count_after"]].describe()
)

print("\nSample before/after:\n")
sample_rows = df_clean.sample(5, random_state=42)

for idx, row in sample_rows.iterrows():
    print("=" * 120)
    print(f"Index: {idx} | Label: {row['label']} | Filename: {row['filename']}")
    print("-" * 120)
    print("BEFORE:")
    print(row["text_before"])
    print("-" * 120)
    print("AFTER:")
    print(row["text"])
    print("=" * 120)

final_df = df_clean[["year", "filename", "text", "word_count", "label", "source_column"]].copy()

output_path = "/kaggle/working/HVA-Qewn-only-hazm-normalized-full.csv"
final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nSaved to:", output_path)
print("Final shape:", final_df.shape)
display(final_df.head())


# test on new data

In [ ]:
# =========================================
# Classic ML pipeline on cleaned HVA dataset
# TF-IDF instead of spaCy embeddings
# Input: Hazm-normalized cleaned CSV
# =========================================

import os
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")

# =========================================
# 1) Path
# =========================================

CLEAN_DATA_PATH = "/kaggle/working/HVA-Qewn-only-hazm-normalized-full.csv"

# If your cleaned file has a different name, use this instead:
# CLEAN_DATA_PATH = "/kaggle/working/HVA-Qewn-only-hazm-normalized.csv"

print("Clean data path exists:", os.path.exists(CLEAN_DATA_PATH), CLEAN_DATA_PATH)

# =========================================
# 2) Load cleaned dataset
# =========================================

df = pd.read_csv(CLEAN_DATA_PATH)

print("\nCleaned dataset shape:", df.shape)
print(df.head())
print("\nColumns:", df.columns.tolist())

# =========================================
# 3) Column setup
# =========================================

TEXT_COL = "text"
LABEL_COL = "label"

if "filename" in df.columns:
    GROUP_COL = "filename"
elif "year" in df.columns:
    GROUP_COL = "year"
else:
    GROUP_COL = None

print("\nText column :", TEXT_COL)
print("Label column:", LABEL_COL)
print("Group column:", GROUP_COL)

# =========================================
# 4) Minimal safety cleaning
# Already Hazm-normalized, so only final safeguards
# =========================================

def clean_text_light(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[\u200b\u200d\u200e\u200f\uFEFF]", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_long = df.copy()

df_long[TEXT_COL] = df_long[TEXT_COL].apply(clean_text_light)

df_long = df_long.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df_long = df_long[df_long[TEXT_COL].astype(str).str.len() > 0].copy()
df_long = df_long.reset_index(drop=True)

print("\n===== Cleaned dataset info =====")
print("Shape      :", df_long.shape)
print("Null texts :", df_long[TEXT_COL].isna().sum())
print("Empty texts:", (df_long[TEXT_COL].astype(str).str.len() == 0).sum())
print("\nLabel distribution:")
print(df_long[LABEL_COL].value_counts(dropna=False))

# =========================================
# 5) Label encoding
# =========================================

def encode_labels_binary(labels):
    labels = pd.Series(labels).astype(str)

    unique_labels = sorted(labels.unique().tolist())
    print("\nOriginal labels:", unique_labels)

    lower_map = {x: x.lower().strip() for x in unique_labels}

    ai_candidates = []
    for original, lowered in lower_map.items():
        if any(key in lowered for key in ["ai", "machine", "generated", "gpt", "chatgpt", "model"]):
            ai_candidates.append(original)

    human_candidates = []
    for original, lowered in lower_map.items():
        if any(key in lowered for key in ["human", "real", "original"]):
            human_candidates.append(original)

    if len(unique_labels) != 2:
        raise ValueError(f"Expected binary labels, but found: {unique_labels}")

    if len(ai_candidates) == 1:
        positive_label = ai_candidates[0]
        y = (labels == positive_label).astype(int).values
        label_mapping = {
            "negative_class_0": [x for x in unique_labels if x != positive_label][0],
            "positive_class_1": positive_label
        }
    else:
        le = LabelEncoder()
        y = le.fit_transform(labels.values)
        label_mapping = {cls: int(idx) for idx, cls in enumerate(le.classes_)}

    return y, label_mapping

y, label_mapping = encode_labels_binary(df_long[LABEL_COL].values)

texts = df_long[TEXT_COL].astype(str).values

print("\nLabel mapping:")
print(label_mapping)

print("\nEncoded label distribution:")
print(pd.Series(y).value_counts().sort_index())

# =========================================
# 6) Group-aware train/test split
# =========================================

if GROUP_COL is not None:
    groups = df_long[GROUP_COL].astype(str).values

    print("\nUnique groups:", len(np.unique(groups)))

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_idx, test_idx = next(gss.split(texts, y, groups=groups))

else:
    train_idx, test_idx = train_test_split(
        np.arange(len(texts)),
        test_size=0.2,
        random_state=42,
        stratify=y
    )

X_train_text = texts[train_idx]
X_test_text = texts[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("\nTrain/test sizes:")
print("Train:", len(X_train_text))
print("Test :", len(X_test_text))

print("\nTrain label distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTest label distribution:")
print(pd.Series(y_test).value_counts().sort_index())

# =========================================
# 7) Models: TF-IDF + classifier
# =========================================

def build_models():
    base_tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=False,
        token_pattern=r"(?u)\b\w+\b"
    )

    models = {
        "TFIDF+MultinomialNB": Pipeline([
            ("tfidf", clone(base_tfidf)),
            ("clf", MultinomialNB(alpha=1.0))
        ]),

        "TFIDF+LinearSVC": Pipeline([
            ("tfidf", clone(base_tfidf)),
            ("clf", LinearSVC(C=1.0, random_state=42))
        ]),

        "TFIDF+RandomForest": Pipeline([
            ("tfidf", clone(base_tfidf)),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "TFIDF+KNN": Pipeline([
            ("tfidf", clone(base_tfidf)),
            ("clf", KNeighborsClassifier(
                n_neighbors=15,
                metric="cosine"
            ))
        ]),
    }

    return models

# =========================================
# 8) Evaluation
# =========================================

def evaluate_model(model, X_train_text, y_train, X_test_text, y_test, name="model"):
    model.fit(X_train_text, y_train)

    y_pred = model.predict(X_test_text)

    y_score = None

    if hasattr(model, "predict_proba"):
        try:
            probs = model.predict_proba(X_test_text)
            if probs.ndim == 2 and probs.shape[1] == 2:
                y_score = probs[:, 1]
        except Exception:
            y_score = None

    if y_score is None and hasattr(model, "decision_function"):
        try:
            scores = model.decision_function(X_test_text)
            if np.ndim(scores) == 1:
                y_score = scores
        except Exception:
            y_score = None

    acc = accuracy_score(y_test, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    cm = confusion_matrix(y_test, y_pred)

    roc_auc = np.nan
    if y_score is not None and len(np.unique(y_test)) == 2:
        try:
            roc_auc = roc_auc_score(y_test, y_score)
        except Exception:
            roc_auc = np.nan

    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    if not np.isnan(roc_auc):
        print(f"ROC-AUC  : {roc_auc:.4f}")
    else:
        print("ROC-AUC  : NaN")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        digits=4,
        zero_division=0
    ))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "cm": cm,
        "y_true": y_test,
        "y_pred": y_pred,
        "trained_model": model
    }

# =========================================
# 9) Run experiments
# =========================================

models = build_models()
results = {}

for model_name, model in models.items():
    results[model_name] = evaluate_model(
        model=clone(model),
        X_train_text=X_train_text,
        y_train=y_train,
        X_test_text=X_test_text,
        y_test=y_test,
        name=f"Hazm-normalized HVA dataset - {model_name}"
    )

# =========================================
# 10) Summarize results
# =========================================

results_df = pd.DataFrame([
    {
        "model": model_name,
        "accuracy": result["accuracy"],
        "precision": result["precision"],
        "recall": result["recall"],
        "f1": result["f1"],
        "roc_auc": result["roc_auc"]
    }
    for model_name, result in results.items()
]).sort_values("f1", ascending=False).reset_index(drop=True)

print("\n===== All Results on Hazm-normalized HVA dataset =====")
display(results_df.round(4))

print("\n===== Best model by F1 =====")
print(results_df.iloc[0].to_dict())

# =========================================
# 11) Save outputs
# =========================================

RESULTS_PATH = "/kaggle/working/classic_ml_results_hazm_normalized.csv"
results_df.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")

print("\nSaved results to:")
print(RESULTS_PATH)
